# Selecting Rows

Often, we would like to extract just those rows that correspond to entries with a particular feature. For example, we might want only the rows corresponding to the Warriors, or to players who earned more than $\$10$ million. Or we might just want the top five earners.

In [1]:
import polars as pl
path_data = '../../../assets/data/'

In [2]:
nba_salaries = pl.read_csv(path_data + 'nba_salaries.csv')
nba = nba_salaries.rename({"'15-'16 SALARY": "SALARY"})

## Specified Rows
The DataFrame method `gather` does just that – it takes a specified set of rows. Its argument is a row index or a sequence of indices, and it creates a new DataFrame consisting of only those rows.

For example, if we wanted just the first row of `nba`, we could use `gather` as follows.

In [3]:
nba

PLAYER,POSITION,TEAM,SALARY
str,str,str,f64
"""Paul Millsap""","""PF""","""Atlanta Hawks""",18.671659
"""Al Horford""","""C""","""Atlanta Hawks""",12.0
"""Tiago Splitter""","""C""","""Atlanta Hawks""",9.75625
"""Jeff Teague""","""PG""","""Atlanta Hawks""",8.0
"""Kyle Korver""","""SG""","""Atlanta Hawks""",5.746479
…,…,…,…
"""Gary Neal""","""PG""","""Washington Wizards""",2.139
"""DeJuan Blair""","""C""","""Washington Wizards""",2.0
"""Kelly Oubre Jr.""","""SF""","""Washington Wizards""",1.92024


In [4]:
nba.gather(0)

PLAYER,POSITION,TEAM,SALARY
str,str,str,f64
"""Paul Millsap""","""PF""","""Atlanta Hawks""",18.671659


This is a new DataFrame with just the single row that we specified.

We could also get the fourth, fifth, and sixth rows by specifying a range of indices as the argument.

In [5]:
nba.gather(range(3, 6))

PLAYER,POSITION,TEAM,SALARY
str,str,str,f64
"""Jeff Teague""","""PG""","""Atlanta Hawks""",8.0
"""Kyle Korver""","""SG""","""Atlanta Hawks""",5.746479
"""Thabo Sefolosha""","""SF""","""Atlanta Hawks""",4.0


If we want a DataFrame of the top 5 highest paid players, we can first sort the rows by salary and then use `head` to keep the first five rows:

In [6]:
nba.sort("SALARY", descending=True).head(5)

PLAYER,POSITION,TEAM,SALARY
str,str,str,f64
"""Kobe Bryant""","""SF""","""Los Angeles Lakers""",25.0
"""Joe Johnson""","""SF""","""Brooklyn Nets""",24.894863
"""LeBron James""","""SF""","""Cleveland Cavaliers""",22.9705
"""Carmelo Anthony""","""SF""","""New York Knicks""",22.875
"""Dwight Howard""","""C""","""Houston Rockets""",22.359364


## Rows Corresponding to a Specified Feature
More often, we will want to access data in a set of rows that have a certain feature, but whose indices we don't know ahead of time. For example, we might want data on all the players who made more than $\$10$ million, but we don't want to spend time counting rows in the sorted DataFrame.

The method `filter` does the job for us. Its output is a DataFrame with the same columns as the original but only the rows where the feature occurs.

`filter` takes a condition built from `pl.col`. If the feature is "made more than $\$10$ million", the column is `SALARY`.

In the first example, we extract the data for all those who earned more than $\$10$ million.

In [7]:
nba.filter(pl.col("SALARY") > 10)

PLAYER,POSITION,TEAM,SALARY
str,str,str,f64
"""Paul Millsap""","""PF""","""Atlanta Hawks""",18.671659
"""Al Horford""","""C""","""Atlanta Hawks""",12.0
"""Joe Johnson""","""SF""","""Brooklyn Nets""",24.894863
"""Thaddeus Young""","""PF""","""Brooklyn Nets""",11.235955
"""Al Jefferson""","""C""","""Charlotte Hornets""",13.5
…,…,…,…
"""DeMar DeRozan""","""SG""","""Toronto Raptors""",10.05
"""Gordon Hayward""","""SF""","""Utah Jazz""",15.40957
"""John Wall""","""PG""","""Washington Wizards""",15.85195


The condition `pl.col("SALARY") > 10` ensured that each selected row had a value of `SALARY` that was greater than 10.

There are 69 rows in the new DataFrame, corresponding to the 69 players who made more than $10$ million dollars. Arranging these rows in order makes the data easier to analyze. DeMar DeRozan of the Toronto Raptors was the "poorest" of this group, at a salary of just over $10$ million dollars.

In [8]:
nba.filter(pl.col("SALARY") > 10).sort("SALARY")

PLAYER,POSITION,TEAM,SALARY
str,str,str,f64
"""DeMar DeRozan""","""SG""","""Toronto Raptors""",10.05
"""Gerald Wallace""","""SF""","""Philadelphia 76ers""",10.105855
"""Luol Deng""","""SF""","""Miami Heat""",10.151612
"""Monta Ellis""","""SG""","""Indiana Pacers""",10.3
"""Wilson Chandler""","""SF""","""Denver Nuggets""",10.449438
…,…,…,…
"""Dwight Howard""","""C""","""Houston Rockets""",22.359364
"""Carmelo Anthony""","""SF""","""New York Knicks""",22.875
"""LeBron James""","""SF""","""Cleveland Cavaliers""",22.9705


How much did Stephen Curry make? For the answer, we have to access the row where the value of `PLAYER` is equal to `Stephen Curry`. That is placed a DataFrame consisting of just one line:

In [9]:
nba.filter(pl.col("PLAYER") == "Stephen Curry")

PLAYER,POSITION,TEAM,SALARY
str,str,str,f64
"""Stephen Curry""","""PG""","""Golden State Warriors""",11.370786


Curry made just under $\$11.4$ million dollars. That's a lot of money, but it's less than half the salary of LeBron James. You'll find that salary in the "Top 5" DataFrame earlier in this section, or you could find it replacing `'Stephen Curry'` by `'LeBron James'` in the line of code above.

Equality is written with `==`, just as `>` selects values above a cutoff. Thus for example you can get a DataFrame of all the Warriors:

In [10]:
with pl.Config(tbl_rows=-1):
    display(nba.filter(pl.col("TEAM") == "Golden State Warriors"))

PLAYER,POSITION,TEAM,SALARY
str,str,str,f64
"""Klay Thompson""","""SG""","""Golden State Warriors""",15.501
"""Draymond Green""","""PF""","""Golden State Warriors""",14.26087
"""Andrew Bogut""","""C""","""Golden State Warriors""",13.8
"""Andre Iguodala""","""SF""","""Golden State Warriors""",11.710456
"""Stephen Curry""","""PG""","""Golden State Warriors""",11.370786
"""Jason Thompson""","""PF""","""Golden State Warriors""",7.008475
"""Shaun Livingston""","""PG""","""Golden State Warriors""",5.543725
"""Harrison Barnes""","""SF""","""Golden State Warriors""",3.873398
"""Marreese Speights""","""C""","""Golden State Warriors""",3.815


This portion of the DataFrame is already sorted by salary, because the original DataFrame listed players sorted by salary within the same team. `pl.Config(tbl_rows=-1)` makes `display` show every row. Without it, the display shows the first five and last five rows.

Equality uses `==` inside `filter`.

In [11]:
nba.filter(pl.col("TEAM") == "Denver Nuggets")

PLAYER,POSITION,TEAM,SALARY
str,str,str,f64
"""Danilo Gallinari""","""SF""","""Denver Nuggets""",14.0
"""Kenneth Faried""","""PF""","""Denver Nuggets""",11.235955
"""Wilson Chandler""","""SF""","""Denver Nuggets""",10.449438
"""JJ Hickson""","""C""","""Denver Nuggets""",5.6135
"""Jameer Nelson""","""PG""","""Denver Nuggets""",4.345
…,…,…,…
"""Joffrey Lauvergne""","""C""","""Denver Nuggets""",1.709719
"""Gary Harris""","""SG""","""Denver Nuggets""",1.58448
"""Nikola Jokic""","""C""","""Denver Nuggets""",1.3


## Multiple Features
You can access rows that have multiple specified features, by using `filter` repeatedly. For example, here is a way to extract all the Point Guards whose salaries were over $\$15$ million.

POLARS_CHANGE: The same rows can be selected in one call, `nba.filter((pl.col("POSITION") == "PG") & (pl.col("SALARY") > 15))`.

In [12]:
nba.filter(pl.col("POSITION") == "PG").filter(pl.col("SALARY") > 15)

PLAYER,POSITION,TEAM,SALARY
str,str,str,f64
"""Derrick Rose""","""PG""","""Chicago Bulls""",20.093064
"""Kyrie Irving""","""PG""","""Cleveland Cavaliers""",16.407501
"""Chris Paul""","""PG""","""Los Angeles Clippers""",21.468695
"""Russell Westbrook""","""PG""","""Oklahoma City Thunder""",16.744218
"""John Wall""","""PG""","""Washington Wizards""",15.85195


## General Form
By now you will have realized that the general way to create a new DataFrame by selecting rows with a given feature is to use `filter` and `pl.col` with the appropriate condition:

`df.filter(pl.col("column_name") <comparison> value)`

In [13]:
nba.filter(pl.col("SALARY").is_between(10, 10.3, closed="left"))

PLAYER,POSITION,TEAM,SALARY
str,str,str,f64
"""Luol Deng""","""SF""","""Miami Heat""",10.151612
"""Gerald Wallace""","""SF""","""Philadelphia 76ers""",10.105855
"""Danny Green""","""SG""","""San Antonio Spurs""",10.0
"""DeMar DeRozan""","""SG""","""Toronto Raptors""",10.05


Notice that the DataFrame above includes Danny Green who made $\$10$ million, but *not* Monta Ellis who made $\$10.3$ million. `is_between` with `closed="left"` includes the left end but not the right.

If we specify a condition that isn't satisfied by any row, we get a DataFrame with column labels but no rows.

In [14]:
nba.filter(pl.col("PLAYER") == "Barack Obama")

PLAYER,POSITION,TEAM,SALARY
str,str,str,f64


## Some More Conditions
Here are some conditions you can use in `filter`. Note that `x` and `y` are numbers, `S` is a string, and `Z` is either a number or a string; you have to specify these depending on the feature you want.

| **Condition** | Description |
|---|---|
| `pl.col("col") == Z` | Equal to `Z` |
| `pl.col("col") > x` | Greater than `x` |
| `pl.col("col") >= x` | Greater than or equal to `x` |
| `pl.col("col") < x` | Less than `x` |
| `pl.col("col") <= x` | Less than or equal to `x` |
| `pl.col("col").is_between(x, y, closed="left")` | Greater than or equal to `x`, and less than `y` |
| `pl.col("col").is_between(x, y, closed="none")` | Greater than `x` and less than `y` |
| `pl.col("col").is_between(x, y, closed="both")` | Greater than or equal to `x`, and less than or equal to `y` |
| `pl.col("col").str.contains(S)` | Contains the string `S` |

You can also negate any of these conditions by putting `~` in front of the comparison:

| **Condition** | Description |
|---|---|
| `pl.col("col") != Z` | Not equal to `Z` |
| `~(pl.col("col") > x)` | Not above `x` |

... and so on. The usual rules of logic apply – for example, "not above x" is the same as "below or equal to x".

We end the section with a series of examples. 

The use of `str.contains` can help save some typing. For example, you can just specify `Warriors` instead of `Golden State Warriors`:

In [15]:
with pl.Config(tbl_rows=-1):
    display(nba.filter(pl.col("TEAM").str.contains("Warriors")))

PLAYER,POSITION,TEAM,SALARY
str,str,str,f64
"""Klay Thompson""","""SG""","""Golden State Warriors""",15.501
"""Draymond Green""","""PF""","""Golden State Warriors""",14.26087
"""Andrew Bogut""","""C""","""Golden State Warriors""",13.8
"""Andre Iguodala""","""SF""","""Golden State Warriors""",11.710456
"""Stephen Curry""","""PG""","""Golden State Warriors""",11.370786
"""Jason Thompson""","""PF""","""Golden State Warriors""",7.008475
"""Shaun Livingston""","""PG""","""Golden State Warriors""",5.543725
"""Harrison Barnes""","""SF""","""Golden State Warriors""",3.873398
"""Marreese Speights""","""C""","""Golden State Warriors""",3.815


You can extract data for all the guards, both Point Guards and Shooting Guards:

In [16]:
nba.filter(pl.col("POSITION").str.contains("G"))

PLAYER,POSITION,TEAM,SALARY
str,str,str,f64
"""Jeff Teague""","""PG""","""Atlanta Hawks""",8.0
"""Kyle Korver""","""SG""","""Atlanta Hawks""",5.746479
"""Dennis Schroder""","""PG""","""Atlanta Hawks""",1.7634
"""Tim Hardaway Jr.""","""SG""","""Atlanta Hawks""",1.30452
"""Jason Richardson""","""SG""","""Atlanta Hawks""",0.947276
…,…,…,…
"""Alan Anderson""","""SG""","""Washington Wizards""",4.0
"""Ramon Sessions""","""PG""","""Washington Wizards""",2.170465
"""Gary Neal""","""PG""","""Washington Wizards""",2.139


You can get all the players who were not Cleveland Cavaliers and had a salary of no less than $\$20$ million:

In [17]:
other_than_Cavs = nba.filter(pl.col("TEAM") != "Cleveland Cavaliers")
other_than_Cavs.filter(~(pl.col("SALARY") < 20))

PLAYER,POSITION,TEAM,SALARY
str,str,str,f64
"""Joe Johnson""","""SF""","""Brooklyn Nets""",24.894863
"""Derrick Rose""","""PG""","""Chicago Bulls""",20.093064
"""Dwight Howard""","""C""","""Houston Rockets""",22.359364
"""Chris Paul""","""PG""","""Los Angeles Clippers""",21.468695
"""Kobe Bryant""","""SF""","""Los Angeles Lakers""",25.0
"""Chris Bosh""","""PF""","""Miami Heat""",22.19273
"""Dwyane Wade""","""SG""","""Miami Heat""",20.0
"""Carmelo Anthony""","""SF""","""New York Knicks""",22.875
"""Kevin Durant""","""SF""","""Oklahoma City Thunder""",20.158622


The same DataFrame can be created in many ways. Here is another, and no doubt you can think of more.

In [18]:
other_than_Cavs.filter(pl.col("SALARY") >= 20)

PLAYER,POSITION,TEAM,SALARY
str,str,str,f64
"""Joe Johnson""","""SF""","""Brooklyn Nets""",24.894863
"""Derrick Rose""","""PG""","""Chicago Bulls""",20.093064
"""Dwight Howard""","""C""","""Houston Rockets""",22.359364
"""Chris Paul""","""PG""","""Los Angeles Clippers""",21.468695
"""Kobe Bryant""","""SF""","""Los Angeles Lakers""",25.0
"""Chris Bosh""","""PF""","""Miami Heat""",22.19273
"""Dwyane Wade""","""SG""","""Miami Heat""",20.0
"""Carmelo Anthony""","""SF""","""New York Knicks""",22.875
"""Kevin Durant""","""SF""","""Oklahoma City Thunder""",20.158622


As you can see, the use of `filter` with `pl.col` gives you great flexibility in accessing rows with features that interest you. Don't hesitate to experiment!